# DeepSets Extrapolation Testing

Test trained DeepSets models on **d=9** (unseen during training) to evaluate extrapolation.

**Goal:** Compare which training split yields the best d=9 accuracy.

In [ ]:
import sys
import json
from pathlib import Path

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/code')
else:
    BASE_PATH = Path('../..')  # code/deepsets/extrapolation -> code/

sys.path.insert(0, str(BASE_PATH))

import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from benchmark_models import DeepSets, DeepSetsModel

# Paths
EXTRAP_DIR = BASE_PATH / "deepsets" / "extrapolation"
MODELS_DIR = EXTRAP_DIR / "models"
RESULTS_DIR = EXTRAP_DIR / "results"
PLOTS_DIR = EXTRAP_DIR / "plots"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load Test Dataset (d=9)

In [ ]:
# Load d=9 test dataset
TEST_D = 9
DATASET_PATH = BASE_PATH / "nn_datasets" / f"d{TEST_D}_baseline_array.pt"

print(f"Loading d={TEST_D} test dataset from: {DATASET_PATH}")

if DATASET_PATH.exists():
    data = torch.load(DATASET_PATH, weights_only=False)
    test_detections = data['detections'].float()
    test_labels = data['labels'].float()
    print(f"Loaded {len(test_labels):,} samples")
    print(f"Detections shape: {test_detections.shape}")
else:
    print(f"ERROR: Dataset not found at {DATASET_PATH}")
    print("Please generate d=9 data first.")

## Load Trained Models

In [ ]:
# Load all trained models
SPLITS = ['equal_333333', 'no_d3', 'd5_heavy', 'd7_only']

models = {}
for split_name in SPLITS:
    model_path = MODELS_DIR / f"{split_name}.pt"
    if model_path.exists():
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)
        
        # Initialize model with saved config
        config = checkpoint['config']
        model = DeepSets(
            nickname=split_name,
            phi_hidden=tuple(config['phi_hidden']),
            rho_hidden=tuple(config['rho_hidden']),
            pool=config['pool'],
            dropout=config.get('dropout', 0.0),
            device=device,
            base_path=BASE_PATH,
        )
        model.model.load_state_dict(checkpoint['state_dict'])
        model.model.eval()
        models[split_name] = model
        print(f"Loaded: {split_name}")
    else:
        print(f"NOT FOUND: {split_name}")

print(f"\nLoaded {len(models)} models")

## Evaluate on d=9

In [ ]:
def evaluate_model(model, detections, labels, d, batch_size=512):
    """Evaluate model accuracy."""
    model.model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for i in tqdm(range(0, len(labels), batch_size), desc="Evaluating", leave=False):
            batch_det = detections[i:i+batch_size].to(device)
            batch_labels = labels[i:i+batch_size].to(device)
            
            preds = model.predict(batch_det, d)
            correct += ((preds > 0.5).float() == batch_labels).sum().item()
            total += len(batch_labels)
    
    return correct / total


# Evaluate all models
results = {}

print(f"\nEvaluating on d={TEST_D}:")
print("-" * 50)

for split_name, model in models.items():
    print(f"\nEvaluating: {split_name}")
    acc = evaluate_model(model, test_detections, test_labels, TEST_D)
    ler = 1.0 - acc
    results[split_name] = {'accuracy': acc, 'ler': ler}
    print(f"  Accuracy: {acc*100:.2f}%")
    print(f"  LER: {ler*100:.2f}%")

## Summary & Comparison

In [ ]:
# Print summary
print("\n" + "="*60)
print(f"EXTRAPOLATION RESULTS (d={TEST_D})")
print("="*60)

# Sort by accuracy
sorted_results = sorted(results.items(), key=lambda x: x[1]['accuracy'], reverse=True)

print(f"\n{'Split':<20} {'Accuracy':>12} {'LER':>12}")
print("-" * 46)
for split_name, metrics in sorted_results:
    print(f"{split_name:<20} {metrics['accuracy']*100:>11.2f}% {metrics['ler']*100:>11.2f}%")

best_split = sorted_results[0][0]
best_acc = sorted_results[0][1]['accuracy']
print(f"\nBest Split: {best_split} ({best_acc*100:.2f}% accuracy)")

In [ ]:
# Save results
results_path = RESULTS_DIR / f"d{TEST_D}_extrapolation_results.json"
with open(results_path, 'w') as f:
    json.dump({
        'test_distance': TEST_D,
        'results': results,
        'best_split': best_split,
        'best_accuracy': best_acc,
    }, f, indent=2)
print(f"\nResults saved to: {results_path}")

## Visualization

In [ ]:
# Bar chart comparison
splits = list(results.keys())
accuracies = [results[s]['accuracy'] * 100 for s in splits]

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#2ecc71' if s == best_split else '#3498db' for s in splits]
bars = ax.bar(splits, accuracies, color=colors, edgecolor='black', linewidth=1.2)

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_xlabel('Training Split', fontsize=12)
ax.set_title(f'DeepSets Extrapolation to d={TEST_D}', fontsize=14, fontweight='bold')

# Add value labels
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / f'd{TEST_D}_extrapolation_comparison.png', dpi=150)
plt.show()

print(f"Plot saved to: {PLOTS_DIR / f'd{TEST_D}_extrapolation_comparison.png'}")